# Code Mode with Monty

[Monty](https://github.com/pydantic/monty) is PydanticAI's sandboxed Python interpreter. Instead of the LLM making 10 sequential tool calls (10 round-trips), it writes **one Python script** that calls your tools in parallel. Monty executes it safely in <1 microsecond.

```
Without Code Mode:               With Code Mode:
  LLM call 1 → tool A              LLM call 1 → writes Python:
  LLM call 2 → tool B                a, b = await gather(
  LLM call 3 → tool A                    tool_a(...),
  LLM call 4 → tool C                    tool_b(...),
  LLM call 5 → combine                )
  LLM call 6 → answer                 c = await tool_c(a + b)
                                       return combine(a, b, c)
  6 round-trips, ~30s               Monty runs it → done
                                     1-2 round-trips, ~10s
```

**Install:** `pip install 'pydantic-ai-harness[code-mode]'`

In [ ]:
!pip install 'pydantic-ai-harness[code-mode]'

## How it works with Cloudflare

- **Monty** runs locally in your Python process (sandboxed, <1μs startup)
- **Workers AI** handles LLM inference (generates the Python code)
- **Browser Run** / other tools run on Cloudflare's edge when called

```
Your code (laptop/server)
  │
  ├─ PydanticAI Agent
  │    ├─ CloudflareProvider → Workers AI (inference)
  │    ├─ CodeMode (Monty)  → in-process (executes LLM's code)
  │    └─ Tools             → Cloudflare edge (Browser Run, etc.)
  │
```

The LLM writes Python. Monty runs it. Tool calls go to Cloudflare. Best of both worlds.

## Example: Parallel web research

Without code mode, browsing 3 pages = 3 LLM calls + 3 tool calls = 6 round-trips.
With code mode, the LLM writes `asyncio.gather(browse(url1), browse(url2), browse(url3))` and Monty runs it in one shot.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from pydantic_ai import Agent
from pydantic_ai_cloudflare import cloudflare_model, BrowserRunToolset

try:
    from pydantic_ai_harness import CodeMode
    HAS_MONTY = True
except ImportError:
    HAS_MONTY = False
    print("Install pydantic-ai-harness[code-mode] first")

if HAS_MONTY:
    model = cloudflare_model()
    agent = Agent(
        model,
        capabilities=[CodeMode()],
        system_prompt=(
            "You are a research assistant. Use the browse tool to read "
            "web pages. When asked to compare multiple pages, write Python "
            "code that uses asyncio.gather to fetch them in parallel."
        ),
    )

    @agent.tool_plain
    async def browse(url: str) -> str:
        """Fetch a web page as clean markdown."""
        ts = BrowserRunToolset(tools=["browse"])
        return await ts._browse(url)

    result = agent.run_sync(
        "Read https://example.com and tell me what it says"
    )
    print(result.output)

## What Monty enables

| Use case | Without Monty | With Monty |
|----------|--------------|------------|
| Browse 3 pages | 6 LLM round-trips | 1-2 round-trips |
| Extract + compare data | 8+ calls | 2 calls |
| Multi-step analysis | Sequential, slow | Parallel, fast |
| Token cost | High (repeated context) | Low (one script) |

Monty runs the LLM's code in a sandbox with:
- No filesystem access
- No network access (except through your tools)
- No environment variable access
- <1μs startup time
- Pause/resume/snapshot support

It's the same technology used by [PydanticAI's code mode](https://ai.pydantic.dev/capabilities/code-mode/) and was built by the Pydantic team.